# INC-004 Steps Silver and Gold Review

This notebook inspects an existing pipeline run against the INC-004 contract (`specs/INC-004.md`): the typed `silver_steps_minute` table, eleven steps quality rules (including the `_unit=steps_per_hour` conversion) and the six steps columns of `gold_clinical_daily`. The supplied database is opened read-only. The idempotency section reruns the pipeline against an isolated copy. Raw inputs are read only.

Generate the run first with `python -m wearable_pipeline run --data-dir data --output-dir output/runs/inc004`. Set `INC004_RUN_DIR` if the output directory differs, `INC004_DATA_DIR` to override `data/`, and `INC004_PREVIOUS_RUN_DIR` (default `output/runs/inc003`, an INC-003 run) to compare the pre-existing gold columns.

## Review Sections
1. Environment, paths and display helpers.
2. Row counts: gold, silver steps and quality results.
3. Silver schema, grain and duplicate-minute flags.
4. Steps rule counts against the profiling findings (M-01, M-02, M-07, M-09) and the rule-without-rejects behavior.
5. `_unit` conversion recomputed from silver, with the per-minute medians behind assumption A-02.
6. Activity levels and the gold daily values recomputed independently from silver.
7. Gold schema and key (5b) and example records per situation (5c, 5d).
8. Gaps and duplicate days named in the profiling findings.
9. Existing gold columns versus a previous run (optional).
10. Rerun on an isolated copy: identical tables and report; raw data unchanged.
11. Acceptance summary with PASS/FAIL/NOT RUN.

Aggregates are displayed, plus a few explicitly selected example gold and silver records (steps columns only); reject raw-value payloads are not printed.

In [1]:
import hashlib
import html
import os
import shutil
import sys
import tempfile
from pathlib import Path

import duckdb
from IPython.display import HTML, display

ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / 'src' / 'wearable_pipeline').is_dir()
)
DATA_DIR = Path(os.environ.get('INC004_DATA_DIR', ROOT / 'data')).resolve()
RUN_DIR = Path(os.environ.get('INC004_RUN_DIR', ROOT / 'output' / 'runs' / 'inc004')).resolve()
PREVIOUS_RUN_DIR = Path(
    os.environ.get('INC004_PREVIOUS_RUN_DIR', ROOT / 'output' / 'runs' / 'inc003')
).resolve()
DATABASE_PATH = RUN_DIR / 'wearable.duckdb'
REPORT_PATH = RUN_DIR / 'quality' / 'report.md'

if not DATABASE_PATH.is_file():
    raise FileNotFoundError(f'DuckDB file not found: {DATABASE_PATH}. Set INC004_RUN_DIR.')
if not DATA_DIR.is_dir():
    raise FileNotFoundError(f'Raw data directory not found: {DATA_DIR}')

connection = duckdb.connect(str(DATABASE_PATH), read_only=True)
acceptance = {}


def render_table(title, columns, rows):
    header = ''.join('<th>' + html.escape(str(column)) + '</th>' for column in columns)
    body = ''.join(
        '<tr>' + ''.join('<td>' + html.escape(str(value)) + '</td>' for value in row) + '</tr>'
        for row in rows
    )
    display(HTML(
        '<h4>' + html.escape(title) + '</h4><table><thead><tr>' + header
        + '</tr></thead><tbody>' + body + '</tbody></table>'
    ))


def show_query(title, sql, parameters=None, db=connection):
    cursor = db.execute(sql, parameters or [])
    render_table(title, [item[0] for item in cursor.description], cursor.fetchall())


def status(ok):
    return 'PASS' if ok else 'FAIL'


report_text = REPORT_PATH.read_text(encoding='utf-8')
run_id = next(
    (line.split(': ', 1)[1] for line in report_text.splitlines() if line.startswith('Run id: ')),
    None,
)
print(f'Database: {DATABASE_PATH}')
print(f'Data directory: {DATA_DIR}')
print(f'Run ID: {run_id}')
print(f'DuckDB version: {duckdb.__version__}')

Database: /home/michaelmadeira/wearable-health-data-pipeline/output/runs/inc004/wearable.duckdb
Data directory: /home/michaelmadeira/wearable-health-data-pipeline/data
Run ID: a826c8397cf5ad8e
DuckDB version: 1.5.6


In [2]:
# Section 2: row counts against the profiled dataset.
# 2,014,950 = 42 x 40,320 + 5 x 40,326 + 3 x 39,960 source records; 30 = 19 earlier rules + 11 steps rules.
expected_counts = {
    'gold_clinical_daily': 1400,
    'silver_steps_minute': 2014950,
    'quality_results': 30,
}
observed_counts = {
    name: connection.execute('SELECT COUNT(*) FROM ' + name).fetchone()[0]
    for name in expected_counts
}
render_table(
    'Expected versus observed row counts',
    ['table', 'expected', 'observed', 'status'],
    [(n, e, observed_counts[n], status(observed_counts[n] == e)) for n, e in expected_counts.items()],
)
acceptance['Row counts match the profiled dataset (no steps record dropped)'] = all(
    observed_counts[n] == e for n, e in expected_counts.items()
)

table,expected,observed,status
gold_clinical_daily,1400,1400,PASS
silver_steps_minute,2014950,2014950,PASS
quality_results,30,30,PASS


In [3]:
# Section 3: silver schema, grain and duplicate-minute flags.
show_query(
    'Silver steps schema',
    '''SELECT ordinal_position, column_name, data_type
       FROM information_schema.columns
       WHERE table_name = 'silver_steps_minute' ORDER BY ordinal_position''',
)
grain = connection.execute(
    '''SELECT COUNT(*) - COUNT(DISTINCT (participant_id, minute_ts)),
              COUNT(*) FILTER (WHERE local_date IS NULL),
              COUNT(*) FILTER (WHERE is_orphan OR is_out_of_window),
              COUNT(*) FILTER (WHERE is_duplicate_minute)
       FROM silver_steps_minute'''
).fetchone()
unflagged_duplicates = connection.execute(
    '''SELECT COUNT(*) FROM silver_steps_minute
       WHERE NOT is_duplicate_minute
         AND (participant_id, minute_ts) IN (
             SELECT participant_id, minute_ts FROM silver_steps_minute
             GROUP BY ALL HAVING COUNT(*) > 1)'''
).fetchone()[0]
render_table(
    'Grain checks (silver keeps one row per source record, so a key can repeat only when flagged)',
    ['check', 'observed', 'status'],
    [
        ('Rows sharing a (participant_id, minute_ts) key with another row', grain[0], 'INFO'),
        ('Rows with a repeated key that are not flagged is_duplicate_minute', unflagged_duplicates,
         status(unflagged_duplicates == 0)),
        ('Rows with NULL local_date', grain[1], status(grain[1] == 0)),
        ('Orphan or out-of-window rows', grain[2], 'INFO'),
        ('Rows flagged is_duplicate_minute', grain[3], 'INFO'),
    ],
)
acceptance['Silver repeats a key only on flagged duplicate rows and every date parses'] = (
    unflagged_duplicates == 0 and grain[1] == 0
)

ordinal_position,column_name,data_type
1,participant_id,VARCHAR
2,local_date,DATE
3,minute_ts,TIMESTAMP
4,steps,INTEGER
5,steps_per_minute,INTEGER
6,activity_level,VARCHAR
7,unit,VARCHAR
8,is_orphan,BOOLEAN
9,is_out_of_window,BOOLEAN
10,is_duplicate_minute,BOOLEAN


check,observed,status
"Rows sharing a (participant_id, minute_ts) key with another row",5767,INFO
Rows with a repeated key that are not flagged is_duplicate_minute,0,PASS
Rows with NULL local_date,0,PASS
Orphan or out-of-window rows,0,INFO
Rows flagged is_duplicate_minute,11575,INFO


In [4]:
# Section 4: steps rule counts versus the profiling findings.
show_query(
    'Steps quality results',
    '''SELECT rule_id, handling, severity, checked_count, failed_count
       FROM quality_results WHERE source = 'steps' ORDER BY rule_id''',
)
failed = dict(connection.execute(
    "SELECT rule_id, failed_count FROM quality_results WHERE source = 'steps'"
).fetchall())
profiled = {
    'STEPS_UNIT_CONVERTED': 120963,
    'STEPS_NEGATIVE': 2,
    'STEPS_TIMESTAMP_NOT_MINUTE_ALIGNED': 28,
}
render_table(
    'Profiled defect counts (M-01 unit records, M-02 negative values, M-09 non-zero seconds)',
    ['rule_id', 'profiled', 'observed', 'status'],
    [(r, e, failed.get(r), status(failed.get(r) == e)) for r, e in profiled.items()],
)
print('STEPS_DUPLICATE_MINUTE_CONFLICT (M-07) observed:', failed.get('STEPS_DUPLICATE_MINUTE_CONFLICT'))
print('STEPS_DUPLICATE_MINUTE_EXACT (M-07) observed:', failed.get('STEPS_DUPLICATE_MINUTE_EXACT'))
unobserved = [
    'STEPS_ORPHAN_PARTICIPANT', 'STEPS_PARTICIPANT_FOLDER_MISMATCH', 'STEPS_TIMESTAMP_UNPARSEABLE',
    'STEPS_TIMESTAMP_OUT_OF_WINDOW', 'STEPS_VALUE_INVALID', 'STEPS_UNIT_UNSUPPORTED',
]
render_table(
    'Rules for defects not observed in profiling (expected 0)',
    ['rule_id', 'observed', 'status'],
    [(r, failed.get(r), status(failed.get(r) == 0)) for r in unobserved],
)

rejects = dict(connection.execute(
    "SELECT rule_id, COUNT(*) FROM quality_rejects WHERE source = 'steps' GROUP BY 1"
).fetchall())
with_rejects = [r for r in failed if r != 'STEPS_UNIT_CONVERTED']
render_table(
    'Reject rows against failed_count (the informational conversion rule writes no rejects)',
    ['rule_id', 'failed_count', 'reject rows', 'status'],
    [(r, failed[r], rejects.get(r, 0), status(failed[r] == rejects.get(r, 0))) for r in sorted(with_rejects)]
    + [('STEPS_UNIT_CONVERTED', failed['STEPS_UNIT_CONVERTED'], rejects.get('STEPS_UNIT_CONVERTED', 0),
        status(rejects.get('STEPS_UNIT_CONVERTED', 0) == 0))],
)
unit_days = connection.execute(
    '''SELECT COUNT(DISTINCT (participant_id, local_date)), COUNT(DISTINCT participant_id)
       FROM silver_steps_minute WHERE quality_flags LIKE '%STEPS_UNIT_CONVERTED%' '''
).fetchone()
render_table(
    'Converted records: participant-days and participants (profiled: 84 and 3)',
    ['participant-days', 'participants', 'status'],
    [(unit_days[0], unit_days[1], status(unit_days == (84, 3)))],
)
acceptance['Profiled M-01, M-02 and M-09 counts reproduced (120,963 / 2 / 28; 84 days, 3 participants)'] = (
    all(failed.get(r) == e for r, e in profiled.items()) and unit_days == (84, 3)
)
acceptance['Unobserved-defect steps rules report zero'] = all(failed.get(r) == 0 for r in unobserved)
acceptance['Rejects match failed counts; the conversion rule writes none'] = (
    all(failed[r] == rejects.get(r, 0) for r in with_rejects) and rejects.get('STEPS_UNIT_CONVERTED', 0) == 0
)

rule_id,handling,severity,checked_count,failed_count
STEPS_DUPLICATE_MINUTE_CONFLICT,withhold_field,high,2014950,7183
STEPS_DUPLICATE_MINUTE_EXACT,flag,medium,2014950,4392
STEPS_NEGATIVE,withhold_field,high,2014950,2
STEPS_ORPHAN_PARTICIPANT,exclude_from_output,medium,2014950,0
STEPS_PARTICIPANT_FOLDER_MISMATCH,exclude_from_output,medium,2014950,0
STEPS_TIMESTAMP_NOT_MINUTE_ALIGNED,flag,medium,2014950,28
STEPS_TIMESTAMP_OUT_OF_WINDOW,exclude_from_output,medium,2014950,0
STEPS_TIMESTAMP_UNPARSEABLE,exclude_from_output,medium,2014950,0
STEPS_UNIT_CONVERTED,convert,low,2014950,120963
STEPS_UNIT_UNSUPPORTED,withhold_field,high,2014950,0


rule_id,profiled,observed,status
STEPS_UNIT_CONVERTED,120963,120963,PASS
STEPS_NEGATIVE,2,2,PASS
STEPS_TIMESTAMP_NOT_MINUTE_ALIGNED,28,28,PASS


STEPS_DUPLICATE_MINUTE_CONFLICT (M-07) observed: 7183
STEPS_DUPLICATE_MINUTE_EXACT (M-07) observed: 4392


rule_id,observed,status
STEPS_ORPHAN_PARTICIPANT,0,PASS
STEPS_PARTICIPANT_FOLDER_MISMATCH,0,PASS
STEPS_TIMESTAMP_UNPARSEABLE,0,PASS
STEPS_TIMESTAMP_OUT_OF_WINDOW,0,PASS
STEPS_VALUE_INVALID,0,PASS
STEPS_UNIT_UNSUPPORTED,0,PASS


rule_id,failed_count,reject rows,status
STEPS_DUPLICATE_MINUTE_CONFLICT,7183,7183,PASS
STEPS_DUPLICATE_MINUTE_EXACT,4392,4392,PASS
STEPS_NEGATIVE,2,2,PASS
STEPS_ORPHAN_PARTICIPANT,0,0,PASS
STEPS_PARTICIPANT_FOLDER_MISMATCH,0,0,PASS
STEPS_TIMESTAMP_NOT_MINUTE_ALIGNED,28,28,PASS
STEPS_TIMESTAMP_OUT_OF_WINDOW,0,0,PASS
STEPS_TIMESTAMP_UNPARSEABLE,0,0,PASS
STEPS_UNIT_UNSUPPORTED,0,0,PASS
STEPS_VALUE_INVALID,0,0,PASS


participant-days,participants,status
84,3,PASS


In [5]:
# Section 5: _unit conversion recomputed from silver (independent of the pipeline code path).
conversion = connection.execute(
    '''SELECT COUNT(*),
              COUNT(*) FILTER (WHERE steps % 60 <> 0),
              COUNT(*) FILTER (WHERE steps_per_minute IS NOT NULL AND steps_per_minute * 60 <> steps),
              COUNT(*) FILTER (WHERE steps_per_minute IS NULL)
       FROM silver_steps_minute WHERE unit = 'steps_per_hour' '''
).fetchone()
unitless_mismatch = connection.execute(
    '''SELECT COUNT(*) FROM silver_steps_minute
       WHERE unit IS NULL AND steps_per_minute IS NOT NULL AND steps_per_minute <> steps'''
).fetchone()[0]
render_table(
    'Records with _unit = steps_per_hour (raw steps and unit text are kept in silver)',
    ['check', 'observed', 'status'],
    [
        ('Unit records', conversion[0], status(conversion[0] == 120963)),
        ('Raw hourly values not divisible by 60', conversion[1], status(conversion[1] == 0)),
        ('Rows where steps_per_minute x 60 differs from raw steps', conversion[2], status(conversion[2] == 0)),
        ('Unit records without a normalized value (withheld duplicates)', conversion[3], 'INFO'),
        ('Unit-less rows where steps_per_minute differs from steps', unitless_mismatch,
         status(unitless_mismatch == 0)),
    ],
)
show_query(
    'Per-minute steps behind assumption A-02 (converted unit records versus the other participants)',
    '''SELECT CASE WHEN unit IS NOT NULL THEN 'converted ' || participant_id
                    WHEN participant_id IN ('P005', 'P033', 'P038') THEN 'unit-less records of unit participants'
                    ELSE 'participants without _unit' END AS grp,
              COUNT(*) AS records, MEDIAN(steps_per_minute) AS median_per_minute,
              ROUND(AVG(steps_per_minute), 2) AS mean_per_minute
       FROM silver_steps_minute WHERE steps_per_minute IS NOT NULL
       GROUP BY ALL ORDER BY 1''',
)
print('The medians are plausible against the dictionary 0-30 steps/minute expectation; they support but do not')
print('prove the unit meaning (A-02 stays unconfirmed by the data provider).')
acceptance['Every hourly value is a multiple of 60 and converts exactly'] = (
    conversion[0] == 120963 and conversion[1] == 0 and conversion[2] == 0 and unitless_mismatch == 0
)

check,observed,status
Unit records,120963,PASS
Raw hourly values not divisible by 60,0,PASS
Rows where steps_per_minute x 60 differs from raw steps,0,PASS
Unit records without a normalized value (withheld duplicates),6,INFO
Unit-less rows where steps_per_minute differs from steps,0,PASS


grp,records,median_per_minute,mean_per_minute
converted P005,40317,1.0,1.91
converted P033,40320,5.0,5.58
converted P038,40320,7.0,7.48
participants without _unit,1886808,5.0,7.34


The medians are plausible against the dictionary 0-30 steps/minute expectation; they support but do not
prove the unit meaning (A-02 stays unconfirmed by the data provider).


In [6]:
# Section 6: activity levels and gold daily values recomputed independently from silver.
show_query(
    'Silver activity levels (0-30 per minute at_rest_to_moderate, above 30 burst, NULL when withheld)',
    '''SELECT activity_level, COUNT(*) AS records,
              MIN(steps_per_minute) AS min_per_minute, MAX(steps_per_minute) AS max_per_minute
       FROM silver_steps_minute GROUP BY 1 ORDER BY 1''',
)
level_violations = connection.execute(
    '''SELECT COUNT(*) FROM silver_steps_minute
       WHERE (activity_level IS NULL) <> (steps_per_minute IS NULL)
          OR (activity_level = 'at_rest_to_moderate' AND NOT (steps_per_minute BETWEEN 0 AND 30))
          OR (activity_level = 'burst' AND NOT (steps_per_minute > 30))'''
).fetchone()[0]
burst_flags = connection.execute(
    '''SELECT COUNT(*) FROM silver_steps_minute
       WHERE activity_level = 'burst' AND quality_flags IS NOT NULL AND quality_flags LIKE '%BURST%' '''
).fetchone()[0]
gold_sums = connection.execute(
    '''SELECT SUM(steps_valid_minute_count), SUM(minutes_rest_to_moderate), SUM(minutes_burst),
              COUNT(*) FILTER (WHERE minutes_rest_to_moderate + minutes_burst <> steps_valid_minute_count)
       FROM gold_clinical_daily'''
).fetchone()
render_table(
    'Level and gold invariants',
    ['check', 'observed', 'status'],
    [
        ('Silver rows violating the level definition', level_violations, status(level_violations == 0)),
        ('Burst rows carrying a burst flag (bursts are not a defect)', burst_flags, status(burst_flags == 0)),
        ('Gold valid minutes / rest-to-moderate / burst', f'{gold_sums[0]} / {gold_sums[1]} / {gold_sums[2]}', 'INFO'),
        ('Gold days where the two level counts do not sum to the valid-minute count', gold_sums[3],
         status(gold_sums[3] == 0)),
    ],
)

# Gold recomputed from silver: one value per participant-minute slot (exact duplicates count once;
# conflicting duplicates are already NULL), in-scope rows only.
gold_diff = connection.execute(
    '''WITH slots AS (
           SELECT participant_id, local_date, date_trunc('minute', minute_ts) AS slot,
                  MAX(steps_per_minute) AS spm
           FROM silver_steps_minute
           WHERE NOT is_orphan AND NOT is_out_of_window AND steps_per_minute IS NOT NULL
           GROUP BY ALL),
       daily AS (
           SELECT participant_id, local_date, SUM(spm)::BIGINT AS total, COUNT(*) AS valid,
                  COUNT(*) FILTER (WHERE spm <= 30) AS rest, COUNT(*) FILTER (WHERE spm > 30) AS burst
           FROM slots GROUP BY ALL)
       SELECT COUNT(*) FROM gold_clinical_daily g
       LEFT JOIN daily d USING (participant_id, local_date)
       WHERE g.total_steps IS DISTINCT FROM d.total
          OR g.steps_valid_minute_count <> COALESCE(d.valid, 0)
          OR g.minutes_rest_to_moderate <> COALESCE(d.rest, 0)
          OR g.minutes_burst <> COALESCE(d.burst, 0)'''
).fetchone()[0]
render_table(
    'Gold step values versus an independent recomputation from silver',
    ['gold rows that differ', 'status'],
    [(gold_diff, status(gold_diff == 0))],
)
acceptance['Activity levels follow their definition and bursts are not flagged'] = (
    level_violations == 0 and burst_flags == 0
)
acceptance['Gold level counts sum to the valid-minute count'] = gold_sums[3] == 0
acceptance['Gold step columns equal an independent recomputation from silver'] = gold_diff == 0

activity_level,records,min_per_minute,max_per_minute
at_rest_to_moderate,1987239,0,30
burst,20526,31,1000000
None,7185,None,None


check,observed,status
Silver rows violating the level definition,0,PASS
Burst rows carrying a burst flag (bursts are not a defect),0,PASS
Gold valid minutes / rest-to-moderate / burst,2005567 / 1985041 / 20526,INFO
Gold days where the two level counts do not sum to the valid-minute count,0,PASS


gold rows that differ,status
0,PASS


In [7]:
# Section 5b (gold schema): gold_clinical_daily schema and primary key.
show_query(
    'Gold schema (six steps columns are appended after the ten survey and six sleep columns)',
    '''SELECT ordinal_position, column_name, data_type
       FROM information_schema.columns
       WHERE table_name = 'gold_clinical_daily' ORDER BY ordinal_position''',
)
show_query(
    'Gold key constraints',
    '''SELECT constraint_type, constraint_column_names
       FROM duckdb_constraints() WHERE table_name = 'gold_clinical_daily' ''',
)
gold_columns = [row[0] for row in connection.execute(
    "SELECT column_name FROM information_schema.columns "
    "WHERE table_name = 'gold_clinical_daily' ORDER BY ordinal_position").fetchall()]
acceptance['Gold has 22 columns ending with the six steps columns'] = (
    len(gold_columns) == 22
    and gold_columns[-6:] == ['total_steps', 'steps_valid_minute_count', 'minutes_rest_to_moderate',
                              'minutes_burst', 'steps_present', 'steps_quality_flags']
)

ordinal_position,column_name,data_type
1,participant_id,VARCHAR
2,local_date,DATE
3,fatigue_score,INTEGER
4,stress_score,INTEGER
5,readiness_score,INTEGER
6,sleep_quality_score,INTEGER
7,survey_present,BOOLEAN
8,survey_row_count,INTEGER
9,survey_valid_score_count,INTEGER
10,survey_quality_flags,VARCHAR


constraint_type,constraint_column_names
PRIMARY KEY,"['participant_id', 'local_date']"
NOT NULL,['participant_id']
NOT NULL,['local_date']


In [8]:
# Section 5c: example gold records (one deterministic example per situation, steps columns plus the key).
steps_columns = (
    'participant_id, local_date, total_steps, steps_valid_minute_count, minutes_rest_to_moderate, '
    'minutes_burst, steps_present, steps_quality_flags'
)
situations = [
    ('Clean steps day', 'steps_present AND steps_quality_flags IS NULL AND steps_valid_minute_count = 1440'),
    ('Day with only converted unit records (M-01)', "steps_quality_flags = 'STEPS_UNIT_CONVERTED'"),
    ('Day with a burst minute and no flag (M-02)', 'minutes_burst > 0 AND steps_quality_flags IS NULL'),
    ('Day with a negative value (M-02)', "steps_quality_flags LIKE '%STEPS_NEGATIVE%'"),
    ('Day with conflicting duplicates (M-07)', "steps_quality_flags LIKE '%STEPS_DUPLICATE_MINUTE_CONFLICT%'"),
    ('Day with a non-aligned timestamp (M-09)', "steps_quality_flags LIKE '%STEPS_TIMESTAMP_NOT_MINUTE_ALIGNED%'"),
    ('Day with partial coverage and no flag (M-08)',
     'steps_valid_minute_count BETWEEN 1 AND 1439 AND steps_quality_flags IS NULL'),
    ('Day without steps records', 'NOT steps_present'),
]
for title, condition in situations:
    cursor = connection.execute(
        'SELECT ' + steps_columns + ' FROM gold_clinical_daily WHERE ' + condition
        + ' ORDER BY participant_id, local_date LIMIT 1'
    )
    rows = cursor.fetchall()
    render_table(title, [item[0] for item in cursor.description],
                 rows or [('no such day in this run',) + ('',) * (len(cursor.description) - 1)])
show_query(
    'Same example day across layers: gold row and its silver records by level',
    '''SELECT g.participant_id, g.local_date, g.total_steps, g.steps_valid_minute_count,
              COUNT(s.minute_ts) AS silver_records,
              COUNT(s.minute_ts) FILTER (WHERE s.unit IS NOT NULL) AS unit_records,
              COUNT(s.minute_ts) FILTER (WHERE s.activity_level = 'burst') AS burst_records
       FROM gold_clinical_daily g
       JOIN silver_steps_minute s USING (participant_id, local_date)
       WHERE g.steps_quality_flags = 'STEPS_UNIT_CONVERTED'
       GROUP BY ALL ORDER BY g.participant_id, g.local_date LIMIT 1''',
)

participant_id,local_date,total_steps,steps_valid_minute_count,minutes_rest_to_moderate,minutes_burst,steps_present,steps_quality_flags
P001,2026-01-08,8523,1440,1437,3,True,None


participant_id,local_date,total_steps,steps_valid_minute_count,minutes_rest_to_moderate,minutes_burst,steps_present,steps_quality_flags
P005,2026-01-08,2695,1440,1440,0,True,STEPS_UNIT_CONVERTED


participant_id,local_date,total_steps,steps_valid_minute_count,minutes_rest_to_moderate,minutes_burst,steps_present,steps_quality_flags
P001,2026-01-08,8523,1440,1437,3,True,None


participant_id,local_date,total_steps,steps_valid_minute_count,minutes_rest_to_moderate,minutes_burst,steps_present,steps_quality_flags
P029,2026-02-04,1008877,1439,1430,9,True,STEPS_NEGATIVE


participant_id,local_date,total_steps,steps_valid_minute_count,minutes_rest_to_moderate,minutes_burst,steps_present,steps_quality_flags
P005,2026-01-10,2727,1439,1439,0,True,"STEPS_DUPLICATE_MINUTE_CONFLICT,STEPS_TIMESTAMP_NOT_MINUTE_ALIGNED,STEPS_UNIT_CONVERTED"


participant_id,local_date,total_steps,steps_valid_minute_count,minutes_rest_to_moderate,minutes_burst,steps_present,steps_quality_flags
P005,2026-01-10,2727,1439,1439,0,True,"STEPS_DUPLICATE_MINUTE_CONFLICT,STEPS_TIMESTAMP_NOT_MINUTE_ALIGNED,STEPS_UNIT_CONVERTED"


participant_id,local_date,total_steps,steps_valid_minute_count,minutes_rest_to_moderate,minutes_burst,steps_present,steps_quality_flags
P020,2026-01-27,3615,1080,1080,0,True,None


participant_id,local_date,total_steps,steps_valid_minute_count,minutes_rest_to_moderate,minutes_burst,steps_present,steps_quality_flags
P010,2026-01-08,None,0,0,0,False,None


participant_id,local_date,total_steps,steps_valid_minute_count,silver_records,unit_records,burst_records
P005,2026-01-08,2695,1440,1440,1440,0


In [9]:
# Section 5d: example silver records (steps columns only; raw-value payloads are not printed).
silver_columns = (
    'participant_id, local_date, minute_ts, steps, steps_per_minute, activity_level, unit, '
    'is_duplicate_minute, quality_flags'
)
silver_situations = [
    ('Converted unit record (raw hourly value kept, per-minute value derived)',
     "quality_flags = 'STEPS_UNIT_CONVERTED' AND activity_level = 'at_rest_to_moderate'"),
    ('Converted unit record that is a burst', "quality_flags = 'STEPS_UNIT_CONVERTED' AND activity_level = 'burst'"),
    ('Unit-less burst record (no flag)', 'unit IS NULL AND activity_level = \'burst\' AND quality_flags IS NULL'),
    ('Negative record (value withheld, raw value kept)', "quality_flags LIKE '%STEPS_NEGATIVE%'"),
    ('Conflicting duplicate record (normalized value withheld)',
     "quality_flags LIKE '%STEPS_DUPLICATE_MINUTE_CONFLICT%' AND unit IS NULL"),
    ('Exact duplicate record (value kept, counted once in gold)',
     "quality_flags LIKE '%STEPS_DUPLICATE_MINUTE_EXACT%'"),
    ('Non-aligned timestamp record (seconds kept, not floored)',
     "quality_flags LIKE '%STEPS_TIMESTAMP_NOT_MINUTE_ALIGNED%'"),
]
for title, condition in silver_situations:
    cursor = connection.execute(
        'SELECT ' + silver_columns + ' FROM silver_steps_minute WHERE ' + condition
        + ' ORDER BY participant_id, minute_ts LIMIT 2'
    )
    rows = cursor.fetchall()
    render_table(title, [item[0] for item in cursor.description],
                 rows or [('no such record in this run',) + ('',) * (len(cursor.description) - 1)])

# Full gold rows, transposed so all 22 columns fit on screen.
for title, condition in [
    ('Gold row with converted unit records, every column', "steps_quality_flags = 'STEPS_UNIT_CONVERTED'"),
    ('Gold row without steps records, every column', 'NOT steps_present'),
]:
    cursor = connection.execute(
        'SELECT * FROM gold_clinical_daily WHERE ' + condition + ' ORDER BY participant_id, local_date LIMIT 2'
    )
    rows = cursor.fetchall()
    columns = [item[0] for item in cursor.description]
    render_table(
        title,
        ['column'] + ['example ' + str(n) for n in range(1, len(rows) + 1)] if rows else ['column'],
        [(name, *(row[i] for row in rows)) for i, name in enumerate(columns)] if rows else [('no such day in this run',)],
    )

participant_id,local_date,minute_ts,steps,steps_per_minute,activity_level,unit,is_duplicate_minute,quality_flags
P005,2026-01-08,2026-01-08 00:00:00,0,0,at_rest_to_moderate,steps_per_hour,False,STEPS_UNIT_CONVERTED
P005,2026-01-08,2026-01-08 00:01:00,0,0,at_rest_to_moderate,steps_per_hour,False,STEPS_UNIT_CONVERTED


participant_id,local_date,minute_ts,steps,steps_per_minute,activity_level,unit,is_duplicate_minute,quality_flags
P033,2026-01-08,2026-01-08 09:03:00,1860,31,burst,steps_per_hour,False,STEPS_UNIT_CONVERTED
P033,2026-01-09,2026-01-09 09:46:00,1860,31,burst,steps_per_hour,False,STEPS_UNIT_CONVERTED


participant_id,local_date,minute_ts,steps,steps_per_minute,activity_level,unit,is_duplicate_minute,quality_flags
P001,2026-01-08,2026-01-08 10:06:00,31,31,burst,None,False,None
P001,2026-01-08,2026-01-08 12:38:00,32,32,burst,None,False,None


participant_id,local_date,minute_ts,steps,steps_per_minute,activity_level,unit,is_duplicate_minute,quality_flags
P029,2026-02-04,2026-02-04 00:30:00,-1,None,None,None,False,STEPS_NEGATIVE
P037,2026-01-16,2026-01-16 00:42:00,-1,None,None,None,False,STEPS_NEGATIVE


participant_id,local_date,minute_ts,steps,steps_per_minute,activity_level,unit,is_duplicate_minute,quality_flags
P005,2026-01-10,2026-01-10 13:55:22,240,None,None,None,True,"STEPS_DUPLICATE_MINUTE_CONFLICT,STEPS_TIMESTAMP_NOT_MINUTE_ALIGNED"
P005,2026-01-14,2026-01-14 08:41:20,180,None,None,None,True,"STEPS_DUPLICATE_MINUTE_CONFLICT,STEPS_TIMESTAMP_NOT_MINUTE_ALIGNED"


participant_id,local_date,minute_ts,steps,steps_per_minute,activity_level,unit,is_duplicate_minute,quality_flags
P005,2026-01-23,2026-01-23 12:40:00,60,1,at_rest_to_moderate,steps_per_hour,True,"STEPS_DUPLICATE_MINUTE_EXACT,STEPS_UNIT_CONVERTED"
P005,2026-01-23,2026-01-23 12:40:46,60,1,at_rest_to_moderate,steps_per_hour,True,"STEPS_DUPLICATE_MINUTE_EXACT,STEPS_TIMESTAMP_NOT_MINUTE_ALIGNED,STEPS_UNIT_CONVERTED"


participant_id,local_date,minute_ts,steps,steps_per_minute,activity_level,unit,is_duplicate_minute,quality_flags
P005,2026-01-10,2026-01-10 13:55:22,240,None,None,None,True,"STEPS_DUPLICATE_MINUTE_CONFLICT,STEPS_TIMESTAMP_NOT_MINUTE_ALIGNED"
P005,2026-01-10,2026-01-10 13:55:22,240,None,None,steps_per_hour,True,"STEPS_DUPLICATE_MINUTE_CONFLICT,STEPS_TIMESTAMP_NOT_MINUTE_ALIGNED,STEPS_UNIT_CONVERTED"


column,example 1,example 2
participant_id,P005,P005
local_date,2026-01-08,2026-01-09
fatigue_score,4,3
stress_score,4,3
readiness_score,7,2
sleep_quality_score,4,2
survey_present,True,True
survey_row_count,1,1
survey_valid_score_count,4,4
survey_quality_flags,None,None


column,example 1,example 2
participant_id,P010,P019
local_date,2026-01-08,2026-01-08
fatigue_score,3,3
stress_score,5,4
readiness_score,7,5
sleep_quality_score,3,2
survey_present,True,True
survey_row_count,1,1
survey_valid_score_count,4,4
survey_quality_flags,None,None


In [10]:
# Section 8: gaps and duplicate days named in the profiling findings.
show_query(
    'Missing day 2026-01-08 (P010, P019, P022, P025): no records, so NULL total and steps_present false',
    '''SELECT participant_id, local_date, total_steps, steps_valid_minute_count, steps_present, steps_quality_flags
       FROM gold_clinical_daily
       WHERE participant_id IN ('P010', 'P019', 'P022', 'P025') AND local_date = DATE '2026-01-08'
       ORDER BY 1''',
)
show_query(
    'Duplicated day 2026-01-10 (same participants): conflicting minutes withheld',
    '''SELECT participant_id, local_date, total_steps, steps_valid_minute_count, steps_quality_flags
       FROM gold_clinical_daily
       WHERE participant_id IN ('P010', 'P019', 'P022', 'P025') AND local_date = DATE '2026-01-10'
       ORDER BY 1''',
)
show_query(
    'Partial days (P020, P028, P041)',
    '''SELECT participant_id, local_date, steps_valid_minute_count, total_steps, steps_quality_flags
       FROM gold_clinical_daily
       WHERE participant_id IN ('P020', 'P028', 'P041') AND steps_valid_minute_count BETWEEN 1 AND 1439
       ORDER BY 1, 2''',
)
show_query(
    'P005 unit-less records at non-zero seconds sharing a slot with hourly records (per-record conflicts)',
    '''SELECT local_date, COUNT(*) AS conflict_records
       FROM silver_steps_minute
       WHERE participant_id = 'P005' AND quality_flags LIKE '%STEPS_DUPLICATE_MINUTE_CONFLICT%'
       GROUP BY 1 ORDER BY 1''',
)
no_records = connection.execute(
    '''SELECT COUNT(*) FROM gold_clinical_daily
       WHERE participant_id IN ('P010', 'P019', 'P022', 'P025') AND local_date = DATE '2026-01-08'
         AND total_steps IS NULL AND steps_valid_minute_count = 0 AND NOT steps_present'''
).fetchone()[0]
withheld = connection.execute(
    '''SELECT COUNT(*) FROM gold_clinical_daily
       WHERE participant_id IN ('P010', 'P019', 'P022', 'P025') AND local_date = DATE '2026-01-10'
         AND steps_valid_minute_count < 1440
         AND steps_quality_flags LIKE '%STEPS_DUPLICATE_MINUTE_CONFLICT%' '''
).fetchone()[0]
partial = connection.execute(
    '''SELECT COUNT(DISTINCT participant_id) FROM gold_clinical_daily
       WHERE participant_id IN ('P020', 'P028', 'P041') AND steps_valid_minute_count BETWEEN 1 AND 1439'''
).fetchone()[0]
acceptance['2026-01-08 gaps show NULL total, count 0 and steps_present false (4 participants)'] = no_records == 4
acceptance['2026-01-10 conflicting duplicates withheld (4 participants)'] = withheld == 4
acceptance['Partial days visible as lower valid-minute counts (3 participants)'] = partial == 3

participant_id,local_date,total_steps,steps_valid_minute_count,steps_present,steps_quality_flags
P010,2026-01-08,None,0,False,None
P019,2026-01-08,None,0,False,None
P022,2026-01-08,None,0,False,None
P025,2026-01-08,None,0,False,None


participant_id,local_date,total_steps,steps_valid_minute_count,steps_quality_flags
P010,2026-01-10,900,520,"STEPS_DUPLICATE_MINUTE_CONFLICT,STEPS_DUPLICATE_MINUTE_EXACT"
P019,2026-01-10,598,541,"STEPS_DUPLICATE_MINUTE_CONFLICT,STEPS_DUPLICATE_MINUTE_EXACT"
P022,2026-01-10,549,584,"STEPS_DUPLICATE_MINUTE_CONFLICT,STEPS_DUPLICATE_MINUTE_EXACT"
P025,2026-01-10,691,531,"STEPS_DUPLICATE_MINUTE_CONFLICT,STEPS_DUPLICATE_MINUTE_EXACT"


participant_id,local_date,steps_valid_minute_count,total_steps,steps_quality_flags
P020,2026-01-27,1080,3615,None
P028,2026-01-15,1080,6696,None
P041,2026-01-29,1080,3605,None


local_date,conflict_records
2026-01-10,3
2026-01-14,4
2026-01-23,2


In [11]:
# Section 9: pre-existing gold columns versus a previous (INC-003) run, when it exists.
existing_columns = ', '.join(gold_columns[:16])
previous_db = PREVIOUS_RUN_DIR / 'wearable.duckdb'
if previous_db.is_file():
    previous = duckdb.connect(str(previous_db), read_only=True)
    try:
        before = previous.execute('SELECT ' + existing_columns + ' FROM gold_clinical_daily ORDER BY ALL').fetchall()
    finally:
        previous.close()
    after = connection.execute('SELECT ' + existing_columns + ' FROM gold_clinical_daily ORDER BY ALL').fetchall()
    identical = before == after
    render_table(
        'Existing 16 gold columns versus previous run',
        ['rows before', 'rows after', 'identical', 'status'],
        [(len(before), len(after), identical, status(identical))],
    )
    acceptance['Existing 16 gold columns identical to the previous run'] = identical
else:
    print('Previous run not found at', previous_db, ': comparison NOT RUN.')
    acceptance['Existing 16 gold columns identical to the previous run'] = None

rows before,rows after,identical,status
1400,1400,True,PASS


In [12]:
# Section 10: rerun on an isolated copy and check raw-data immutability against the run manifest.
# Tables are compared inside DuckDB (EXCEPT ALL in both directions) to avoid loading 2 million rows into Python.
connection.close()
sys.path.insert(0, str(ROOT / 'src'))
from wearable_pipeline.pipeline import run_pipeline

table_names = [
    'raw_manifest', 'silver_participants', 'silver_wellness_survey', 'silver_sleep_sessions',
    'silver_sleep_stages', 'silver_steps_minute', 'quality_results', 'quality_rejects',
    'gold_clinical_daily',
]
rerun_comparisons = []
with tempfile.TemporaryDirectory(prefix='inc004-idempotency-review-') as temporary_root:
    temporary_root = Path(temporary_root)
    baseline_dir = temporary_root / 'baseline'
    rerun_dir = temporary_root / 'rerun'
    for directory in (baseline_dir, rerun_dir):
        (directory / 'quality').mkdir(parents=True)
        shutil.copy2(DATABASE_PATH, directory / 'wearable.duckdb')
        shutil.copy2(REPORT_PATH, directory / 'quality' / 'report.md')

    rerun_result = run_pipeline(DATA_DIR, rerun_dir)
    compare = duckdb.connect(':memory:')
    compare.execute("ATTACH '" + str(baseline_dir / 'wearable.duckdb') + "' AS before_db (READ_ONLY)")
    compare.execute("ATTACH '" + str(rerun_dir / 'wearable.duckdb') + "' AS after_db (READ_ONLY)")
    try:
        for table_name in table_names:
            rows_before = compare.execute('SELECT COUNT(*) FROM before_db.' + table_name).fetchone()[0]
            only_before = compare.execute(
                'SELECT COUNT(*) FROM (SELECT * FROM before_db.' + table_name
                + ' EXCEPT ALL SELECT * FROM after_db.' + table_name + ')').fetchone()[0]
            only_after = compare.execute(
                'SELECT COUNT(*) FROM (SELECT * FROM after_db.' + table_name
                + ' EXCEPT ALL SELECT * FROM before_db.' + table_name + ')').fetchone()[0]
            rerun_comparisons.append((table_name, rows_before, only_before == 0 and only_after == 0))
    finally:
        compare.close()
    report_identical = (baseline_dir / 'quality' / 'report.md').read_bytes() == (
        rerun_dir / 'quality' / 'report.md'
    ).read_bytes()
    run_id_identical = rerun_result.run_id == run_id

render_table(
    'Same-input rerun comparison',
    ['table', 'rows before', 'identical after rerun', 'status'],
    [(*row, status(row[2])) for row in rerun_comparisons],
)
render_table(
    'Rerun determinism',
    ['check', 'result', 'status'],
    [
        ('Run ID unchanged', run_id_identical, status(run_id_identical)),
        ('Quality report bytes unchanged', report_identical, status(report_identical)),
    ],
)
acceptance["Rerun preserves every table's logical contents"] = all(row[2] for row in rerun_comparisons)
acceptance['Rerun preserves the deterministic run id and report'] = run_id_identical and report_identical

check_db = duckdb.connect(str(DATABASE_PATH), read_only=True)
manifest = {
    row[0]: row[1]
    for row in check_db.execute('SELECT relative_path, sha256 FROM raw_manifest').fetchall()
}
check_db.close()
hash_mismatches = [
    path for path, digest in manifest.items()
    if hashlib.sha256((DATA_DIR / path).read_bytes()).hexdigest() != digest
]
print('Raw files differing from the run manifest SHA-256:', hash_mismatches[:5] or 'none')
acceptance['Raw files match the run manifest hashes'] = not hash_mismatches

table,rows before,identical after rerun,status
raw_manifest,154,True,PASS
silver_participants,50,True,PASS
silver_wellness_survey,1414,True,PASS
silver_sleep_sessions,1400,True,PASS
silver_sleep_stages,7000,True,PASS
silver_steps_minute,2014950,True,PASS
quality_results,30,True,PASS
quality_rejects,14401,True,PASS
gold_clinical_daily,1400,True,PASS


check,result,status
Run ID unchanged,True,PASS
Quality report bytes unchanged,True,PASS


Raw files differing from the run manifest SHA-256: none


In [13]:
# Section 11: acceptance summary.
rows = [
    (name, 'PASS' if result is True else 'FAIL' if result is False else 'NOT RUN')
    for name, result in acceptance.items()
]
rows.append(('Changed-input updates affect exactly one gold row', 'NOT RUN here; covered by the pipeline tests'))
render_table('INC-004 review summary', ['acceptance check', 'status'], rows)
print('Review is observational; only the isolated temporary database copy was rerun.')

acceptance check,status
Row counts match the profiled dataset (no steps record dropped),PASS
Silver repeats a key only on flagged duplicate rows and every date parses,PASS
"Profiled M-01, M-02 and M-09 counts reproduced (120,963 / 2 / 28; 84 days, 3 participants)",PASS
Unobserved-defect steps rules report zero,PASS
Rejects match failed counts; the conversion rule writes none,PASS
Every hourly value is a multiple of 60 and converts exactly,PASS
Activity levels follow their definition and bursts are not flagged,PASS
Gold level counts sum to the valid-minute count,PASS
Gold step columns equal an independent recomputation from silver,PASS
Gold has 22 columns ending with the six steps columns,PASS


Review is observational; only the isolated temporary database copy was rerun.
